# Anchor-Scarce GNN Localisation: All-Variant Model Ablation & Convergence Benchmark
### **6,000 Training Graphs | 80/10/10 Ratio | Multi-Variant Convergence Tracking**
**Target Venue:** IEEE Signal Processing Letters (SPL)

This standalone Colab / Kaggle notebook is specifically designed to **train, diagnose, and benchmark all 5 model ablation variants** of our proposed framework:

| Variant | Architecture Description | Role in Paper Ablation |
| :--- | :--- | :--- |
| **`learned`** | AnchorGNN Encoder + Unrolled SMACOF with learned gating MLP & learned $\alpha$ | **Proposed Model** |
| **`uniform`** | AnchorGNN Encoder + Unrolled SMACOF with uniform weights $w_{ij} = 1$ | Tests classical dwMDS unrolling |
| **`huber`** | AnchorGNN Encoder + Unrolled SMACOF with classical Huber IRLS weights | Tests whether neural gating beats classical IRLS |
| **`huber_alpha`** | AnchorGNN Encoder + Huber IRLS weights + learned step size $\alpha$ | Disentangles learned $\alpha$ from learned weights |
| **`encoder_only`**| Stage-A AnchorGNN predictions ($T = 0$ SMACOF steps) | Isolates GNN contribution without unrolled refinement |

---
### Key Capabilities & Rigorous Design:
1. **Exact 80 / 10 / 10 Split (7,500 Total Graphs):**
   - **Train:** $N_{\text{train}} = 6,000$ graphs ($80.0\%$)
   - **Validation:** $N_{\text{val}} = 750$ graphs ($10.0\%$)
   - **Test:** $N_{\text{test}} = 750$ graphs ($10.0\%$ across missingness levels $10\% - 90\%$)
   All models are trained and tested on the **exact same identical graphs** for 100% scientific fairness.
2. **Automated Multi-Variant Training Loop:** Automatically trains all 5 variants in sequence, tracking per-epoch loss, certified validation RMSE, and early stopping convergence.
3. **Convergence Epoch Detection:** Discovers the exact epoch ($E_{\text{conv}}$) and cumulative graph passes ($E_{\text{conv}} \times 6000$) where validation RMSE reaches its minimum, answering the exact training budget needed.
4. **4 Multi-Variant Publication Figures:** Generates side-by-side comparison curves (Loss, Val RMSE, Test RMSE vs Missingness, Grouped Bar Charts) and saves the underlying CSVs.
5. **Guaranteed Kaggle Output Persistence:** Automatically saves all `.pt` checkpoints, figures, and CSV tables directly to `/kaggle/working/` so they appear permanently in the Kaggle Output sidebar.


In [ ]:
# GPU Verification and Kaggle / Colab Environment Detection
import os, sys, subprocess, shutil, time, warnings
import torch
warnings.filterwarnings('ignore', category=UserWarning, module='torch.autograd')
if hasattr(torch.autograd.graph, 'set_warn_on_accumulate_grad_stream_mismatch'):
    torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)

assert torch.cuda.is_available(), 'No GPU detected! In Colab: Runtime > Change runtime type > GPU. In Kaggle: Settings > Accelerator > GPU T4 x2.'
num_gpus = torch.cuda.device_count()
print(f'GPUs available: {num_gpus}')
for i in range(num_gpus):
    vram_gib = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gib:.1f} GiB VRAM)')
if num_gpus >= 2:
    print(f'\n[+] DUAL-GPU ACCELERATION ACTIVE: Utilizing both GPU 0 and GPU 1 in parallel!')
else:
    print(f'\n[+] Single GPU active: {torch.cuda.get_device_name(0)}')

# Automatic Environment Detection
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in str(sys.modules) or 'COLAB_GPU' in os.environ

if IS_KAGGLE:
    WORKDIR = '/kaggle/working'
    print('\n[+] KAGGLE ENVIRONMENT DETECTED!')
    print(f'[+] Root working directory: {WORKDIR}')
    print('[+] All checkpoints, CSVs, and plots will appear in Kaggle Output sidebar.')
elif IS_COLAB:
    USE_DRIVE = False  # Set to True to persist directly into Google Drive
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            WORKDIR = '/content/drive/MyDrive/gnn_loc_ablation_6000'
        except Exception as e:
            print(f'Drive mount warning: {e}')
            WORKDIR = '/content/gnn_loc_ablation_6000'
    else:
        WORKDIR = '/content/gnn_loc_ablation_6000'
    print(f'\n[+] GOOGLE COLAB DETECTED! Working directory: {WORKDIR}')
else:
    WORKDIR = os.path.abspath('./gnn_loc_ablation_6000')
    print(f'\n[+] LOCAL ENVIRONMENT DETECTED! Working directory: {WORKDIR}')

os.makedirs(WORKDIR, exist_ok=True)
for sub in ['checkpoints', 'results', 'plots', 'data']:
    os.makedirs(os.path.join(WORKDIR, sub), exist_ok=True)

%cd {WORKDIR}
print(f'Active Working Directory: {os.getcwd()}')


In [ ]:
# ---------------- Experiment Settings & Hyperparameters ----------------
# Exact 80 / 10 / 10 dataset split (7,500 graphs total):
N_TRAIN                 = 6000    # 80.0% of total dataset (User requested 6,000 graphs)
N_VAL                   = 750     # 10.0% of total dataset
N_TEST                  = 750     # 10.0% of total dataset (partitioned across 10%-90% missing)

# Training dynamics:
MAX_EPOCHS              = 50      # Maximum epochs to train each variant
EARLY_STOPPING_PATIENCE = 12      # Stop variant if validation does not improve for 12 epochs
MIN_DELTA               = 1e-4    # Minimum certified validation RMSE improvement (meters)
BATCH_GRAPHS            = 32      # 6000 / 32 = 188 gradient updates per epoch
LEARNING_RATE           = 1e-3    # AdamW initial learning rate
WEIGHT_DECAY            = 1e-5    # L2 regularization
GRAD_CLIP               = 1.0     # Maximum gradient norm
HUBER_DELTA             = 0.05    # Normalized Huber loss threshold
AUX_WEIGHT              = 0.3     # Auxiliary supervision weight on Stage-A encoder output

# All 5 model ablation variants to train & evaluate:
VARIANTS = ['learned', 'uniform', 'huber', 'huber_alpha', 'encoder_only']

# Polish & transfer toggles:
EVALUATE_HUBER_POLISH   = True    # Also evaluate refined positions with Huber polish
RUN_TRANSFER_TEST       = True    # Evaluate cross-topology transfer on best learned model
SEED                    = 1234    # Master seed for exact scientific reproducibility
# -----------------------------------------------------------------------

print('Experiment Configuration Locked:')
print(f'  Dataset Budget:     {N_TRAIN} Train (80%) | {N_VAL} Val (10%) | {N_TEST} Test (10%)')
print(f'  Variants Scheduled: {VARIANTS}')
print(f'  Max Epochs / Early Stop: {MAX_EPOCHS} epochs / patience {EARLY_STOPPING_PATIENCE}')


## Step 1: Core Mathematical & Geometric Building Blocks
The cells below write `config.py`, `common.py`, `model.py`, and `step1_data.py` into the environment.
These modules implement:
- Canonical anchor frame (strictly $E(2)$ invariant)
- Slepian-Bangs CRLB multiplicative noise formulation
- Exact Huber-IRLS starting point (zero-initialized gating linear layer)
- 80/10/10 dataset generation routines for 6,000 training graphs


In [ ]:
%%writefile config.py
"""
config.py -- experiment settings.
"""
import os

N_NODES        = 50
N_ANCHORS      = 3
COORD_RANGE    = 10.0

GRAPH_TYPES    = ["random", "rgg", "ba"]
MISSING_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

NOISE_MODELS = {
    "clean": dict(sigma=0.00, p_nlos=0.0, nlos_max=0.0),
    "gauss": dict(sigma=0.05, p_nlos=0.0, nlos_max=0.0),
    "nlos":  dict(sigma=0.02, p_nlos=0.2, nlos_max=4.0),
}

# 80 / 10 / 10 ratio:
N_TRAIN          = 6000
N_VAL            = 750
N_TEST           = 750
TRAIN_N_RANGE    = (30, 80)
TRAIN_MISSING    = (0.10, 0.90)

TEST_SCENARIOS   = [("random", "clean"), ("random", "gauss"), ("random", "nlos"),
                    ("rgg", "clean"),    ("rgg", "gauss"),    ("rgg", "nlos"),
                    ("ba", "clean"),     ("ba", "nlos")]

HIDDEN          = 96
ENC_LAYERS      = 5
REFINE_STEPS    = 10
TEST_REFINE_STEPS = 50
HUBER_DELTA     = 0.05
AUX_WEIGHT      = 0.3

EPOCHS          = 50
BATCH_GRAPHS    = 32
LR              = 1e-3
WEIGHT_DECAY    = 1e-5
GRAD_CLIP       = 1.0
SEED            = 1234

RUN_TRANSFER        = True
TRANSFER_TRAIN_SETS = ["random", "rgg", "ba", "random+rgg", "random+ba", "rgg+ba", "all"]
TRANSFER_MIN_MISSING = 0.5

def train_types(key):
    types = list(GRAPH_TYPES) if key == "all" else key.split("+")
    return types

ROOT        = os.getcwd()
DATA_DIR    = os.path.join(ROOT, "data")
CKPT_DIR    = os.path.join(ROOT, "checkpoints")
RESULTS_DIR = os.path.join(ROOT, "results")
PLOTS_DIR   = os.path.join(ROOT, "plots")

for _d in (DATA_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR):
    os.makedirs(_d, exist_ok=True)


In [ ]:
%%writefile common.py
"""
common.py -- shared building blocks used by step1 / step2 / step3.

    1. geometry           : points, distance matrices, E(2) canonical anchor frame
    2. graph generators   : random (spanning tree + extra edges), RGG (MST-first), BA
    3. noise              : multiplicative Gaussian + positive NLOS bias
    4. anchors            : well-spread anchor selection
    5. rigidity           : is the answer even unique?  (graph + per node)
    6. features           : E(2)-equivariant features (supports any K >= 3 anchors)
    7. metrics            : coordinate RMSE, certified RMSE, relative EDM error
"""
import os
import numpy as np
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, minimum_spanning_tree


# =============================================================================
# 1. Geometry & E(2) Canonical Frame
# =============================================================================
def random_points(n, rng, coord_range=10.0):
    """n points uniform in [-R, R]^2, shape (n, 2)."""
    return rng.uniform(-coord_range, coord_range, size=(n, 2))


def dist_matrix(P):
    """Raw (NOT squared) Euclidean distance matrix of points P (n, 2)."""
    diff = P[:, None, :] - P[None, :, :]
    return np.sqrt((diff ** 2).sum(-1))


def best_anchor_triangle(anchor_xy):
    """If K > 3 anchors are given, pick the 3 anchors with the largest triangle area
    to define a well-conditioned canonical coordinate frame."""
    K = len(anchor_xy)
    if K == 3:
        return np.array([0, 1, 2], dtype=np.int64)
    best_tri, best_area = (0, 1, 2), -1.0
    for a in range(K):
        for b in range(a + 1, K):
            for c in range(b + 1, K):
                u = anchor_xy[b] - anchor_xy[a]
                v = anchor_xy[c] - anchor_xy[a]
                area = abs(u[0] * v[1] - u[1] * v[0]) * 0.5
                if area > best_area:
                    best_area = area
                    best_tri = (a, b, c)
    return np.array(best_tri, dtype=np.int64)


def canonical_anchor_frame(anchor_xy):
    """
    Constructs an exact E(2)-equivariant canonical coordinate frame from K >= 3 anchors:
      1. Translation: origin = centroid c of the 3 reference anchors
      2. Scale: unit = mean pairwise distance s of the 3 reference anchors
      3. Rotation: align vector (anchor_1 - anchor_0) with positive x-axis
    Reference anchors are canonically ordered by intrinsic triangle geometry
    (vertex opposite longest side first, then next-longest), ensuring complete
    invariance under rotation, reflection, and arbitrary node indexing.
    Returns (c, s, R, tri) where R is a (2, 2) orthogonal matrix in O(2) such that:
      X_can = ((X_orig - c) / s) @ R
      X_orig = (X_can @ R.T) * s + c
    """
    tri = best_anchor_triangle(anchor_xy)
    A3 = anchor_xy[tri].astype(np.float64)
    c = A3.mean(0)
    # Order reference anchors by intrinsic triangle geometry (invariant to rotation, reflection and relabelling):
    # Vertex opposite the longest side first, then next-longest.
    # Note: ambiguous for exactly isosceles triangles (measure zero for generic random coordinates).
    opp = np.array([np.linalg.norm(A3[1] - A3[2]),
                    np.linalg.norm(A3[0] - A3[2]),
                    np.linalg.norm(A3[0] - A3[1])])
    order = np.argsort(-opp)
    tri, A3 = tri[order], A3[order]

    s = np.mean([np.linalg.norm(A3[i] - A3[j]) for i in range(3) for j in range(i + 1, 3)])
    s = max(float(s), 1e-6)
    A_s = (A3 - c) / s
    u = A_s[1] - A_s[0]
    norm_u = np.linalg.norm(u)
    e1 = u / norm_u if norm_u > 1e-9 else np.array([1.0, 0.0])
    e2 = np.array([-e1[1], e1[0]])
    if np.dot(A_s[2] - A_s[0], e2) < 0:
        e2 = -e2
    R = np.column_stack([e1, e2])
    return c.astype(np.float32), np.float32(s), R.astype(np.float32), tri


# =============================================================================
# 2. Graph generators  -- all return an (m, 2) array of undirected edges i<j
# =============================================================================
def _target_edges(n, missing):
    total = n * (n - 1) // 2
    return max(n - 1, int(round(total * (1.0 - missing))))


def graph_random(n, missing, rng):
    """
    "Spanning-tree" generator.
      1. random spanning tree: node order is shuffled and each node attaches
         to a uniformly random earlier node   -> connected, n-1 edges
      2. add uniformly random extra pairs until the target edge count.
    """
    order = rng.permutation(n)
    edges = set()
    for k in range(1, n):
        a, b = order[k], order[rng.integers(0, k)]
        edges.add((min(a, b), max(a, b)))
    target = _target_edges(n, missing)
    iu = np.array(np.triu_indices(n, 1)).T
    rng.shuffle(iu)
    for a, b in iu:
        if len(edges) >= target:
            break
        edges.add((int(a), int(b)))
    return np.array(sorted(edges), dtype=np.int64)


def graph_rgg(P, missing):
    """
    Random geometric (radio-range) graph:
      1. Start with the Euclidean minimum spanning tree (n-1 edges) so the graph
         is guaranteed connected.
      2. Add the shortest remaining pairs until the exact target edge count is reached.
    """
    n = len(P)
    D = dist_matrix(P)
    mst = minimum_spanning_tree(csr_matrix(D)).tocoo()
    edges = set()
    for a, b in zip(mst.row, mst.col):
        edges.add((int(min(a, b)), int(max(a, b))))
    target = _target_edges(n, missing)
    iu = np.triu_indices(n, 1)
    order = np.argsort(D[iu])
    for idx in order:
        if len(edges) >= target:
            break
        edges.add((int(iu[0][idx]), int(iu[1][idx])))
    return np.array(sorted(edges), dtype=np.int64)


def ba_m_for(n, missing):
    """m whose BA edge count m*(n-m) is closest to the target for this missing %."""
    target = _target_edges(n, missing)
    ms = np.arange(1, n // 2 + 1)
    return int(ms[np.argmin(np.abs(ms * (n - ms) - target))])


def ba_reachable(n, missing, tol=0.05):
    """True if a BA graph can match this missing % to within tol."""
    m = ba_m_for(n, missing)
    total = n * (n - 1) // 2
    return abs((1.0 - m * (n - m) / total) - missing) <= tol


def graph_ba(n, m, rng):
    """Barabasi-Albert graph: each new node attaches to m existing nodes."""
    G = nx.barabasi_albert_graph(n, m, seed=int(rng.integers(1 << 31)))
    e = np.array([(min(a, b), max(a, b)) for a, b in G.edges()], dtype=np.int64)
    return e


def is_connected(n, edges):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(edges.tolist())
    return nx.is_connected(G)


# =============================================================================
# 3. Noise (on RAW distances)
# =============================================================================
def noisy_distances(d_true, rng, sigma=0.0, p_nlos=0.0, nlos_max=0.0, return_mask=False):
    """d_obs = d (1 + sigma*eps) + b * U[0, nlos_max],  b ~ Bernoulli(p_nlos)."""
    d = d_true * (1.0 + sigma * rng.standard_normal(d_true.shape))
    is_nlos = np.zeros(d_true.shape, dtype=bool)
    if p_nlos > 0:
        mask = rng.random(d_true.shape) < p_nlos
        bias = rng.uniform(0, nlos_max, d_true.shape) * mask
        d = d + bias
        is_nlos = mask & (bias > 0)
    d_out = np.maximum(d, 1e-3)
    if return_mask:
        return d_out, is_nlos
    return d_out



# =============================================================================
# 4. Anchors
# =============================================================================
def pick_anchors(P, k, rng, tries=60):
    """k anchors, chosen among `tries` random k-subsets to maximise the
    smallest triangle area (well-spread, never collinear)."""
    n = len(P)
    best, best_score = None, -1.0
    for _ in range(tries):
        idx = rng.choice(n, size=k, replace=False)
        A = P[idx]
        score = np.inf
        for a in range(k):
            for b in range(a + 1, k):
                for c in range(b + 1, k):
                    u, v = A[b] - A[a], A[c] - A[a]
                    score = min(score, abs(u[0] * v[1] - u[1] * v[0]) / 2)
        if score > best_score:
            best, best_score = idx, score
    return np.sort(best)


# =============================================================================
# 5. Rigidity: is the answer unique?
# =============================================================================
def _trilaterate_from(Adj, seed):
    """Grow a set from `seed`: add any node with >= 3 neighbours in the set."""
    done = np.zeros(len(Adj), bool)
    done[list(seed)] = True
    while True:
        new = (Adj @ done >= 3) & ~done
        if not new.any():
            return done
        done |= new


def trilateration_certified(n, edges, anchors, max_seeds=3000, rng=None):
    """Per-node SUFFICIENT certificate of unique localisability (generic positions)."""
    Adj = np.zeros((n, n), np.int32)
    Adj[edges[:, 0], edges[:, 1]] = 1
    Adj[edges[:, 1], edges[:, 0]] = 1
    anchors = list(anchors)
    cert = np.zeros(n, bool)
    comp = _trilaterate_from(Adj, anchors)
    cert |= comp
    tris = []
    for i, j in edges:
        for k in np.nonzero(Adj[i] & Adj[j])[0]:
            if k > j:
                tris.append((i, j, k))
    if rng is not None and len(tris) > max_seeds:
        tris = [tris[t] for t in rng.choice(len(tris), max_seeds, replace=False)]
    seen = np.zeros(n, bool)
    for t in tris:
        if seen[list(t)].all():
            continue
        comp = _trilaterate_from(Adj, t)
        seen |= comp
        if comp[anchors].all():
            cert |= comp
    return cert


def globally_rigid(n, edges, anchors, rng, d=2, tol=1e-8):
    """Randomised generic GLOBAL rigidity test (Gortler-Healy-Thurston 2010)."""
    e = set(map(tuple, edges.tolist()))
    for i in range(len(anchors)):
        for j in range(i + 1, len(anchors)):
            a, b = sorted((int(anchors[i]), int(anchors[j])))
            e.add((a, b))
    E = np.array(sorted(e))
    q = rng.standard_normal((n, d))
    R = np.zeros((len(E), d * n))
    for k, (i, j) in enumerate(E):
        diff = q[i] - q[j]
        R[k, d * i:d * i + d] = diff
        R[k, d * j:d * j + d] = -diff
    s = np.linalg.svd(R, compute_uv=False)
    rankR = int((s > tol * s[0]).sum())
    if rankR < d * n - d * (d + 1) // 2:
        return False
    z = rng.standard_normal(len(E))
    w = z - R @ np.linalg.lstsq(R, z, rcond=None)[0]
    Om = np.zeros((n, n))
    for k, (i, j) in enumerate(E):
        Om[i, j] -= w[k]
        Om[j, i] -= w[k]
        Om[i, i] += w[k]
        Om[j, j] += w[k]
    ev = np.abs(np.linalg.eigvalsh(Om))
    rankOm = int((ev > 1e-7 * max(ev.max(), 1e-12)).sum())
    return rankOm == n - d - 1


# =============================================================================
# 6. Features  -- built ONLY from observed distances + anchor coordinates
# =============================================================================
def geodesic_matrix(n, edges, d_obs, indices=None):
    """Shortest-path lengths through the observed graph.
    If indices is provided, calculates single-source shortest paths only from indices,
    returning shape (len(indices), n). Otherwise calculates full (n, n) matrix."""
    W = csr_matrix((np.r_[d_obs, d_obs], (np.r_[edges[:, 0], edges[:, 1]],
                                         np.r_[edges[:, 1], edges[:, 0]])), shape=(n, n))
    return shortest_path(W, directed=False, indices=indices)


def hop_matrix(n, edges, rows):
    W = csr_matrix((np.ones(2 * len(edges)), (np.r_[edges[:, 0], edges[:, 1]],
                                              np.r_[edges[:, 1], edges[:, 0]])), shape=(n, n))
    return shortest_path(W, directed=False, unweighted=True, indices=rows)


def multilaterate(anchor_xy, g):
    """Linearised least-squares multilateration (DV-distance style)."""
    a1 = anchor_xy[0]
    A = 2.0 * (anchor_xy[1:] - a1)                                  # (K-1, 2)
    b = ((anchor_xy[1:] ** 2).sum(1) - (a1 ** 2).sum())[None, :] \
        - g[:, 1:] ** 2 + g[:, :1] ** 2                             # (n, K-1)
    X, *_ = np.linalg.lstsq(A, b.T, rcond=None)
    return X.T


def build_features(n, edges, d_obs, anchors, anchor_xy):
    """
    Returns a dict of arrays in an E(2)-CANONICALIZED frame:
        origin = anchor centroid (c), unit = mean pairwise anchor distance (s),
        orientation = canonical O(2) basis R from the 3 best-spread anchors (tri).
    Node features (19, invariant to K >= 3 anchors):
        is_anchor (1) | own anchor xy in canonical frame (2) | x0 multilateration init (2)
        geodesic dists to reference anchor triangle (3) | canonical anchor triangle xy (6)
        hop counts to reference anchor triangle / 10 (3) | log(1+degree) (1) | mean edge length (1)
    Edge features (3, per DIRECTED edge):
        d_obs | d_obs^2 | |d_obs - ||x0_i - x0_j|||   (last one flags NLOS-like edges)
    """
    c, s, R, tri = canonical_anchor_frame(anchor_xy)
    A_n = ((anchor_xy - c) / s) @ R
    d_n = d_obs / s

    G_all = geodesic_matrix(n, edges, d_n, indices=anchors).T       # (n, K)
    H_all = hop_matrix(n, edges, anchors).T                         # (n, K)
    x0 = multilaterate(A_n, G_all)
    x0[anchors] = A_n

    G_feat = G_all[:, tri]
    H_feat = H_all[:, tri]
    A_tri = A_n[tri]

    deg = np.bincount(edges.ravel(), minlength=n).astype(float)
    sum_len = np.bincount(edges[:, 0], weights=d_n, minlength=n) + \
              np.bincount(edges[:, 1], weights=d_n, minlength=n)
    is_a = np.zeros(n)
    is_a[anchors] = 1
    own = np.zeros((n, 2))
    own[anchors] = A_n
    X = np.concatenate([
        is_a[:, None], own, x0, G_feat,
        np.repeat(A_tri.reshape(1, -1), n, 0),
        H_feat / 10.0, np.log1p(deg)[:, None], (sum_len / np.maximum(deg, 1))[:, None],
    ], axis=1)

    src = np.r_[edges[:, 0], edges[:, 1]]
    dst = np.r_[edges[:, 1], edges[:, 0]]
    dd = np.r_[d_n, d_n]
    incons = np.abs(dd - np.linalg.norm(x0[src] - x0[dst], axis=1))
    EF = np.stack([dd, dd ** 2, incons], 1)
    return dict(x=X.astype(np.float32), x0=x0.astype(np.float32),
                edge_index=np.stack([src, dst]).astype(np.int64),
                edge_attr=EF.astype(np.float32), d_edge=dd.astype(np.float32),
                center=c.astype(np.float32), scale=np.float32(s),
                rot=R.astype(np.float32))


# =============================================================================
# One complete graph sample
# =============================================================================
def make_graph(rng, n, graph_type, missing, noise, n_anchors=3, coord_range=10.0,
               ba_m=None, check_global=True):
    """
    Build one sample. Everything a method may use lives under 'obs';
    ground truth lives under 'true_xy' (training loss + scoring only).
    """
    P = random_points(n, rng, coord_range)
    if graph_type == "random":
        edges = graph_random(n, missing, rng)
    elif graph_type == "rgg":
        edges = graph_rgg(P, missing)
    elif graph_type == "ba":
        if ba_m is None:                  # main comparison: m chosen from the missing %
            ba_m = ba_m_for(n, missing)
        edges = graph_ba(n, ba_m, rng)
    else:
        raise ValueError(graph_type)
    d_true = np.linalg.norm(P[edges[:, 0]] - P[edges[:, 1]], axis=1)
    d_obs, is_nlos = noisy_distances(d_true, rng, return_mask=True, **noise)
    anchors = pick_anchors(P, n_anchors, rng)
    feats = build_features(n, edges, d_obs, anchors, P[anchors])
    total = n * (n - 1) // 2
    gr = globally_rigid(n, edges, anchors, rng) if check_global else None
    cert = np.ones(n, bool) if gr else trilateration_certified(n, edges, anchors, rng=rng)
    return dict(
        n=n, graph_type=graph_type, missing_nominal=float(missing),
        missing_actual=1.0 - len(edges) / total,
        edges=edges, d_obs=d_obs.astype(np.float32),
        is_nlos=is_nlos.astype(bool),
        anchors=anchors, anchor_xy=P[anchors].astype(np.float32),
        true_xy=P.astype(np.float32),
        certified=cert,            # nodes whose position is provably unique
        globally_rigid=gr,         # whole graph uniquely localisable?
        ba_m=ba_m,                 # None unless graph_type == "ba"
        **feats,
    )



# =============================================================================
# 7. Metrics (ground truth used HERE only)
# =============================================================================
def unknown_mask(g):
    m = np.ones(g["n"], bool)
    m[g["anchors"]] = False
    return m


def coord_rmse(g, X, mask=None):
    """RMSE over non-anchor nodes (or a sub-mask of them)."""
    m = unknown_mask(g) if mask is None else (mask & unknown_mask(g))
    if m.sum() == 0:
        return np.nan
    return float(np.sqrt(((X[m] - g["true_xy"][m]) ** 2).sum(1).mean()))


def rel_edm_error(g, X):
    """||D_hat - D||_F / ||D||_F on raw distances (blind to reflections)."""
    D, Dh = dist_matrix(g["true_xy"]), dist_matrix(X)
    return float(np.linalg.norm(D - Dh) / np.linalg.norm(D))


def cramer_rao_lower_bound(g, sigma=None):
    """
    Computes the theoretical Cramér-Rao Lower Bound (CRLB) on coordinate RMSE
    under additive Gaussian range noise (Patwari et al., IEEE SPM 2005).
    FIM J is of size 2(N-K) x 2(N-K) over unknown sensor nodes.
    Returns sqrt( (1 / (N-K)) * Tr(J^{-1}) ) in coordinate units (meters),
    or NaN if the network is non-rigid / rank-deficient.
    """
    n = g["n"]
    anchors = set(g["anchors"])
    P = g["true_xy"]
    edges = g["edges"]
    unk = [k for k in range(n) if k not in anchors]
    m = len(unk)
    if m == 0:
        return 0.0
    idx_map = {node: k for k, node in enumerate(unk)}
    diff_all = P[edges[:, 0]] - P[edges[:, 1]]
    d_true = np.linalg.norm(diff_all, axis=1)

    if sigma is None:
        nz = g.get("noise")
        sig_frac = nz.get("sigma", 0.05) if isinstance(nz, dict) else 0.05
    else:
        sig_frac = sigma

    FIM = np.zeros((2 * m, 2 * m), dtype=np.float64)
    for (u_node, v_node), d_ij in zip(edges, d_true):
        if d_ij < 1e-9:
            continue
        sig_ij = max(sig_frac * d_ij, 1e-4)
        # Slepian-Bangs formula under multiplicative range noise d_obs ~ N(d, sigma^2 d^2):
        # differentiating both mean and covariance gives w = (1 + 2*sigma^2) / (sigma^2 d^2)
        w = (1.0 + 2.0 * (sig_frac ** 2)) / (sig_ij ** 2)
        u_vec = (P[u_node] - P[v_node]) / d_ij
        uuT = np.outer(u_vec, u_vec) * w

        i_unk = u_node in idx_map
        j_unk = v_node in idx_map
        if i_unk and j_unk:
            idx_i, idx_j = idx_map[u_node], idx_map[v_node]
            FIM[2*idx_i:2*idx_i+2, 2*idx_i:2*idx_i+2] += uuT
            FIM[2*idx_j:2*idx_j+2, 2*idx_j:2*idx_j+2] += uuT
            FIM[2*idx_i:2*idx_i+2, 2*idx_j:2*idx_j+2] -= uuT
            FIM[2*idx_j:2*idx_j+2, 2*idx_i:2*idx_i+2] -= uuT
        elif i_unk and not j_unk:
            idx_i = idx_map[u_node]
            FIM[2*idx_i:2*idx_i+2, 2*idx_i:2*idx_i+2] += uuT
        elif not i_unk and j_unk:
            idx_j = idx_map[v_node]
            FIM[2*idx_j:2*idx_j+2, 2*idx_j:2*idx_j+2] += uuT

    try:
        evals, evecs = np.linalg.eigh(FIM)
        if evals.min() <= 1e-7:
            return np.nan
        inv_FIM = evecs @ np.diag(1.0 / evals) @ evecs.T
        return float(np.sqrt(np.trace(inv_FIM) / m))
    except Exception:
        return np.nan



def dump_run_config(out_dir=None, step_name=None):
    """
    Dumps all resolved configuration parameters, git commit hash, environment,
    hardware specs, and library versions to results/config_used.json for reproducibility.
    """
    import os, json, sys, platform, subprocess, datetime
    import config as C
    out_dir = out_dir or C.RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)

    cfg_dict = {}
    for k in dir(C):
        if not k.startswith("_") and k.isupper():
            val = getattr(C, k)
            if isinstance(val, (int, float, str, bool, list, dict, tuple)):
                cfg_dict[k] = val
            else:
                cfg_dict[k] = str(val)

    git_hash = "unknown"
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            git_hash = r.stdout.strip()
    except Exception:
        pass

    packages = {
        "python": sys.version,
        "platform": platform.platform(),
    }
    for pkg in ("torch", "numpy", "scipy", "pandas", "matplotlib", "cvxpy"):
        try:
            mod = __import__(pkg)
            packages[pkg] = getattr(mod, "__version__", "installed")
        except ImportError:
            packages[pkg] = "not installed"

    hardware = {
        "cpu_count": os.cpu_count(),
    }
    try:
        import torch
        hardware["cuda_available"] = torch.cuda.is_available()
        hardware["cuda_device_count"] = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if torch.cuda.is_available():
            hardware["devices"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        pass

    info = {
        "timestamp": datetime.datetime.now().isoformat(),
        "step": step_name or "general",
        "git_commit": git_hash,
        "hardware": hardware,
        "packages": packages,
        "config": cfg_dict,
    }

    target_path = os.path.join(out_dir, "config_used.json")
    with open(target_path, "w") as f:
        json.dump(info, f, indent=2)
    if step_name:
        step_path = os.path.join(out_dir, f"config_used_{step_name}.json")
        with open(step_path, "w") as f:
            json.dump(info, f, indent=2)
    print(f"  [Config Dump] Saved run configuration & environment -> {target_path}", flush=True)
    return info


In [ ]:
%%writefile model.py
"""
model.py -- AnchorGNN + learned unrolled SMACOF.
Supports all 5 variants: learned, uniform, huber, huber_alpha, encoder_only.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

EPS = 1e-6

def scatter_sum(src, index, n):
    out = torch.zeros((n,) + src.shape[1:], dtype=src.dtype, device=src.device)
    return out.index_add_(0, index, src)

def scatter_mean(src, index, n):
    s = scatter_sum(src, index, n)
    c = scatter_sum(torch.ones_like(src[:, :1]), index, n).clamp_min(1.0)
    return s / c

def mlp(i, h, o, layers=2):
    mods, d = [], i
    for _ in range(layers - 1):
        mods += [nn.Linear(d, h), nn.SiLU()]
        d = h
    mods.append(nn.Linear(d, o))
    return nn.Sequential(*mods)

def collate(graphs, device="cpu"):
    xs, x0s, eis, eas, des, ys, am, certs, batch, cen, sca, rots = [], [], [], [], [], [], [], [], [], [], [], []
    off = 0
    for b, g in enumerate(graphs):
        n = g["n"]
        R = g.get("rot", np.eye(2, dtype=np.float32))
        xs.append(g["x"]); x0s.append(g["x0"])
        eis.append(g["edge_index"] + off); eas.append(g["edge_attr"]); des.append(g["d_edge"])
        if "y_norm" in g:
            ys.append(g["y_norm"])
        else:
            ys.append(((g["true_xy"] - g["center"]) / g["scale"]) @ R)
        if "anchor_mask" in g:
            am.append(g["anchor_mask"])
        else:
            a = np.zeros(n, bool); a[g["anchors"]] = True; am.append(a)
        if "certified" in g:
            certs.append(g["certified"])
        else:
            certs.append(np.ones(n, bool))
        batch.append(np.full(n, b)); cen.append(g["center"]); sca.append(g["scale"]); rots.append(R)
        off += n
    T = lambda a, dt=torch.float32: torch.as_tensor(np.concatenate(a), dtype=dt, device=device)
    return dict(
        x=T(xs), x0=T(x0s), edge_index=torch.as_tensor(np.concatenate(eis, 1), device=device),
        edge_attr=T(eas), d_edge=T(des), y=T(ys), anchor_mask=T(am, torch.bool),
        certified=T(certs, torch.bool),
        batch=T(batch, torch.long), center=torch.as_tensor(np.stack(cen), device=device),
        scale=torch.as_tensor(np.array(sca), device=device),
        rot=torch.as_tensor(np.stack(rots), device=device),
        sizes=[g["n"] for g in graphs],
    )

class MPLayer(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.msg = mlp(3 * H, H, H)
        self.upd = mlp(2 * H, H, H)
        self.ln_h, self.ln_e = nn.LayerNorm(H), nn.LayerNorm(H)

    def forward(self, h, e, ei):
        src, dst = ei
        m = self.msg(torch.cat([h[dst], h[src], e], -1))
        e = self.ln_e(e + m)
        agg = scatter_mean(m, dst, h.size(0))
        h = self.ln_h(h + self.upd(torch.cat([h, agg], -1)))
        return h, e

class LearnedSMACOFStep(nn.Module):
    def __init__(self, H, variant="learned", huber_delta=0.05):
        super().__init__()
        self.variant, self.delta = variant, huber_delta
        self.w_static = nn.Linear(3 * H, H)
        self.w_dyn = nn.Linear(3, H, bias=False)
        self.w_out = nn.Linear(H, 1)
        nn.init.zeros_(self.w_out.weight)
        nn.init.zeros_(self.w_out.bias)
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def prepare(self, h, e, ei):
        if self.variant != "learned":
            return None
        src, dst = ei
        return self.w_static(torch.cat([h[dst], h[src], e], -1))

    def weights(self, static, d, dhat, r):
        if self.variant == "uniform":
            return torch.ones_like(d)
        if self.variant in ("huber", "huber_alpha"):
            return torch.clamp(self.delta / r.abs().clamp_min(EPS), max=1.0)
        z = F.silu(static + self.w_dyn(torch.stack([d, dhat, r.abs()], -1)))
        w_base = torch.clamp(self.delta / r.abs().clamp_min(EPS), max=1.0)
        w_gate = torch.sigmoid(self.w_out(z)).squeeze(-1) * 2.0
        return (w_base * w_gate).clamp(min=1e-3, max=1.0)

    def forward(self, x, static, ei, d, anchor_mask, x_anchor):
        src, dst = ei
        diff = x[dst] - x[src]
        dhat = diff.norm(dim=-1).clamp_min(1e-4)
        r = d - dhat
        w = self.weights(static, d, dhat, r)
        target = x[src] + d[:, None] * diff / dhat[:, None]
        num = scatter_sum(w[:, None] * target, dst, x.size(0))
        den = scatter_sum(w[:, None], dst, x.size(0)).clamp_min(EPS)
        learn_alpha = (self.variant in ("learned", "huber_alpha"))
        alpha = torch.sigmoid(self.alpha_logit).clamp(min=0.1, max=0.7) if learn_alpha else 0.5
        step_update = (num / den - x).clamp(min=-5.0, max=5.0)
        x_new = x + alpha * step_update
        return torch.where(anchor_mask[:, None], x_anchor, x_new), w

class AnchorGNNLocalizer(nn.Module):
    def __init__(self, in_dim=19, edge_dim=3, H=96, enc_layers=5, refine_steps=10,
                 variant="learned", huber_delta=0.05):
        super().__init__()
        self.variant = variant
        self.delta = huber_delta
        self.T = 0 if variant == "encoder_only" else refine_steps
        self.node_in = mlp(in_dim, H, H)
        self.edge_in = mlp(edge_dim, H, H)
        self.layers = nn.ModuleList([MPLayer(H) for _ in range(enc_layers)])
        self.head = mlp(H, H, 2)
        self.step = LearnedSMACOFStep(H, variant, huber_delta)

    def forward(self, B, T=None, return_all=False):
        x0, am, ei, d = B["x0"], B["anchor_mask"], B["edge_index"], B["d_edge"]
        x_anchor = x0
        h = self.node_in(B["x"])
        e = self.edge_in(B["edge_attr"])
        for L in self.layers:
            h, e = L(h, e, ei)
        xA = torch.where(am[:, None], x_anchor, x0 + self.head(h))
        x, traj = xA, [xA]
        static = self.step.prepare(h, e, ei)
        for _ in range(self.T if T is None else T):
            x, _ = self.step(x, static, ei, d, am, x_anchor)
            traj.append(x)
        return (x, xA, traj) if return_all else (x, xA)

def to_original_units(B, x_norm):
    out, off = [], 0
    for b, n in enumerate(B["sizes"]):
        R = B["rot"][b]
        xb = (x_norm[off:off + n] @ R.T) * B["scale"][b] + B["center"][b]
        out.append(xb.detach().cpu().numpy()); off += n
    return out

def supervised_loss(x, B, delta):
    m = ~B["anchor_mask"]
    if "certified" in B:
        m_cert = m & B["certified"]
        if m_cert.any():
            m = m_cert
    return F.huber_loss(x[m], B["y"][m], delta=delta)


In [ ]:
%%writefile step1_data.py
"""
step1_data.py -- generate 6000 train, 750 val, 750 test graphs (80/10/10 split).
"""
import pickle, time, os
import numpy as np
import config as C
from common import make_graph

def save(obj, name):
    path = os.path.join(C.DATA_DIR, name)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  saved {len(obj):5d} graphs -> {path}")

def mixed_set(rng, count, types=None, label=""):
    types = types or C.GRAPH_TYPES
    out = []
    noise_names = list(C.NOISE_MODELS)
    log_step = max(1, count // 5)
    t0 = time.time()
    while len(out) < count:
        n = int(rng.integers(C.TRAIN_N_RANGE[0], C.TRAIN_N_RANGE[1] + 1))
        gt = types[rng.integers(len(types))]
        lo, hi = C.TRAIN_MISSING
        if gt == "ba":
            lo = max(lo, 0.5)
        mp = float(rng.uniform(lo, hi))
        nz = noise_names[rng.integers(len(noise_names))]
        g = make_graph(rng, n, gt, mp, C.NOISE_MODELS[nz], C.N_ANCHORS, C.COORD_RANGE)
        g["noise"] = nz
        out.append(g)
        if len(out) % log_step == 0 or len(out) == count:
            print(f"    ... {label}: {len(out)}/{count} graphs ({len(out)/count*100:.0f}%) in {time.time()-t0:.1f}s", flush=True)
    return out

def scenario_set(rng, count=750):
    scenarios = C.TEST_SCENARIOS
    levels = C.MISSING_LEVELS
    out = []
    per_level = max(1, count // len(levels))
    print(f"  Generating balanced test set: {count} graphs across {len(levels)} missing levels (~{per_level} per level)...")
    for mp in levels:
        valid_scens = [s for s in scenarios if not (s[0] == "ba" and mp < 0.5)]
        for k in range(per_level):
            gt, nz = valid_scens[k % len(valid_scens)]
            g = make_graph(rng, C.N_NODES, gt, mp, C.NOISE_MODELS[nz], C.N_ANCHORS, C.COORD_RANGE)
            g.update(noise=nz, scenario=f"{gt}/{nz}", graph_id=len(out))
            out.append(g)
            if len(out) >= count:
                break
        if len(out) >= count:
            break
    while len(out) < count:
        gt, nz = scenarios[rng.integers(len(scenarios))]
        mp = float(rng.choice(levels))
        if gt == "ba" and mp < 0.5: mp = 0.5
        g = make_graph(rng, C.N_NODES, gt, mp, C.NOISE_MODELS[nz], C.N_ANCHORS, C.COORD_RANGE)
        g.update(noise=nz, scenario=f"{gt}/{nz}", graph_id=len(out))
        out.append(g)
    return out

if __name__ == "__main__":
    t0 = time.time()
    print("STEP 1: Building datasets with exact 80/10/10 split (6000 Train / 750 Val / 750 Test)...")
    print("  [1/4] Generating Training set (6,000 graphs)...")
    save(mixed_set(np.random.default_rng(C.SEED), C.N_TRAIN, label="train"), "train.pkl")
    print("  [2/4] Generating Validation set (750 graphs)...")
    save(mixed_set(np.random.default_rng(C.SEED + 1), C.N_VAL, label="val"), "val.pkl")
    print("  [3/4] Generating Test set (750 graphs across 10%-90% missing)...")
    test_graphs = scenario_set(np.random.default_rng(C.SEED + 2), count=C.N_TEST)
    save(test_graphs, "test.pkl")

    print("  [4/4] Generating Generalisation (N=100, 200) and Barabasi-Albert datasets...")
    rng_gen = np.random.default_rng(C.SEED + 3)
    for n in [100, 200]:
        gen = [make_graph(rng_gen, n, "rgg", mp, C.NOISE_MODELS["nlos"], C.N_ANCHORS, C.COORD_RANGE)
               for mp in [0.5, 0.7, 0.9] for _ in range(10)]
        save(gen, f"test_n{n}.pkl")

    rng_ba = np.random.default_rng(C.SEED + 4)
    ba = []
    for m in [1, 2, 3, 4, 5]:
        for k in range(10):
            g = make_graph(rng_ba, C.N_NODES, "ba", 0.0, C.NOISE_MODELS["clean"],
                           C.N_ANCHORS, C.COORD_RANGE, ba_m=m)
            g.update(noise="clean", scenario=f"ba/m={m}", ba_m=m, graph_id=k)
            ba.append(g)
    save(ba, "test_ba.pkl")

    if C.RUN_TRANSFER:
        print("  [5/5] Generating transfer pools for cross-topology experiment...")
        for i, gt in enumerate(C.GRAPH_TYPES):
            save(mixed_set(np.random.default_rng(C.SEED + 10 + i), C.N_TRAIN, [gt], label=f"train_{gt}"), f"train_pool_{gt}.pkl")
            save(mixed_set(np.random.default_rng(C.SEED + 20 + i), C.N_VAL, [gt], label=f"val_{gt}"), f"val_pool_{gt}.pkl")

    print(f"STEP 1 done in {time.time() - t0:.1f}s")


## Step 2: Build & Verify the Datasets (80/10/10 Split)
Builds `train.pkl` (6,000 graphs), `val.pkl` (750 graphs), and `test.pkl` (750 graphs).
If the datasets already exist in `data/`, generation is safely skipped.


In [ ]:
# Execute dataset generation or verify existing files
if not (os.path.exists('data/train.pkl') and os.path.exists('data/val.pkl') and os.path.exists('data/test.pkl')):
    print('[+] Generating 6000 train, 750 val, and 750 test graphs...')
    subprocess.run([sys.executable, 'step1_data.py'], check=True)
else:
    print('[+] Datasets already exist on disk.')

import pickle, pandas as pd
with open('data/train.pkl', 'rb') as f: train_g = pickle.load(f)
with open('data/val.pkl', 'rb') as f:   val_g   = pickle.load(f)
with open('data/test.pkl', 'rb') as f:  test_g  = pickle.load(f)

print(f'\n[+] Dataset Partition Verified:')
print(f'    Train Set: {len(train_g):5d} graphs ({len(train_g)/(len(train_g)+len(val_g)+len(test_g))*100:.1f}%)')
print(f'    Val Set:   {len(val_g):5d} graphs ({len(val_g)/(len(train_g)+len(val_g)+len(test_g))*100:.1f}%)')
print(f'    Test Set:  {len(test_g):5d} graphs ({len(test_g)/(len(train_g)+len(val_g)+len(test_g))*100:.1f}%)')
print(f'    Total:     {len(train_g)+len(val_g)+len(test_g):5d} graphs (Exact 80 / 10 / 10 Split)')


## Step 3: Training & Convergence Diagnostic Engine
This engine trains any requested variant with:
- **AdamW Optimizer & Cosine Warmup Learning Rate Schedule**
- **Huber Loss with Auxiliary Stage-A Supervision**
- **Per-Epoch Certified Validation RMSE Tracking**
- **Automatic Early Stopping & Convergence Epoch Detection**
- **Checkpoint Saving:** Best model weights saved to `checkpoints/best_model_{variant}.pt`
- **History Logging:** Per-epoch loss, val RMSE, learning rate, and elapsed time saved to `results/history_{variant}.csv`


In [ ]:
import os, time, random, pickle
import numpy as np, pandas as pd
import torch
import torch.nn as nn
from model import AnchorGNNLocalizer, collate, supervised_loss, to_original_units
from common import coord_rmse
import config as C

def polish_coordinates(g, X0):
    """Fast local Huber polish for model predictions (no baseline solvers needed)."""
    from scipy.optimize import minimize
    n, E, d = g['n'], g['edges'], g['d_obs'].astype(float)
    I, J = E[:, 0], E[:, 1]
    free = np.ones(n, bool); free[g['anchors']] = False
    Xfix = X0.copy(); Xfix[g['anchors']] = g['anchor_xy']
    delta = 0.1 * np.median(d)
    def f(z):
        X = Xfix.copy(); X[free] = z.reshape(-1, 2)
        diff = X[I] - X[J]
        dh = np.sqrt((diff ** 2).sum(1) + 1e-12)
        r = dh - d
        a = np.abs(r)
        val = np.where(a <= delta, 0.5 * r ** 2, delta * (a - 0.5 * delta)).sum()
        dr = np.where(a <= delta, r, delta * np.sign(r))
        c = (dr / dh)[:, None] * diff
        grad = np.zeros_like(X)
        np.add.at(grad, I, c); np.add.at(grad, J, -c)
        return val, grad[free].ravel()
    res = minimize(f, X0[free].ravel(), method='L-BFGS-B', jac=True,
                   options=dict(maxiter=100, ftol=1e-8))
    X_out = Xfix.copy(); X_out[free] = res.x.reshape(-1, 2)
    return X_out

def evaluate_fast(model, graphs, device, bs=64, refine_steps=50, model1=None, dev1=None):
    model.eval()
    errs, errs_cert = [], []
    T_steps = None if model.variant == 'encoder_only' else refine_steps
    if model1 is not None and dev1 is not None and len(graphs) >= 2:
        import concurrent.futures
        model1.eval()
        mid = len(graphs) // 2
        g0, g1 = graphs[:mid], graphs[mid:]

        def run_chunk(m, chk, dev):
            local_errs, local_cert = [], []
            with torch.no_grad():
                for i in range(0, len(chk), bs):
                    b_chk = chk[i:i + bs]
                    B = collate(b_chk, dev)
                    x, _ = m(B, T=T_steps)
                    for g, X in zip(b_chk, to_original_units(B, x)):
                        local_errs.append(coord_rmse(g, X))
                        local_cert.append(coord_rmse(g, X, g.get('certified')))
            return local_errs, local_cert

        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            f0 = pool.submit(run_chunk, model, g0, device)
            f1 = pool.submit(run_chunk, model1, g1, dev1)
            e0, c0 = f0.result()
            e1, c1 = f1.result()
            errs = e0 + e1
            errs_cert = c0 + c1
    else:
        with torch.no_grad():
            for i in range(0, len(graphs), bs):
                chunk = graphs[i:i + bs]
                B = collate(chunk, device)
                x, _ = model(B, T=T_steps)
                for g, X in zip(chunk, to_original_units(B, x)):
                    errs.append(coord_rmse(g, X))
                    errs_cert.append(coord_rmse(g, X, g.get('certified')))
    return float(np.nanmean(errs_cert)), float(np.nanmean(errs)), float(np.nanmedian(errs))

def train_variant(variant_name='learned', train_on=None, train_graphs=None, val_graphs=None, device='cuda:0',
                  max_epochs=50, patience=12, min_delta=1e-4, batch_size=32):
    if train_on:
        types = C.train_types(train_on)
        train_graphs, val_graphs = [], []
        for gt in types:
            with open(f'data/train_pool_{gt}.pkl', 'rb') as f:
                train_graphs += pickle.load(f)[:C.N_TRAIN // len(types)]
            with open(f'data/val_pool_{gt}.pkl', 'rb') as f:
                val_graphs += pickle.load(f)[:C.N_VAL // len(types)]
        model_tag = f'{variant_name}__train-{train_on}'
    else:
        model_tag = variant_name
        if train_graphs is None:
            with open('data/train.pkl', 'rb') as f: train_graphs = pickle.load(f)
        if val_graphs is None:
            with open('data/val.pkl', 'rb') as f: val_graphs = pickle.load(f)

    for g in train_graphs + val_graphs:
        if 'y_norm' not in g:
            g['y_norm'] = (((g['true_xy'] - g['center']) / g['scale']) @ g.get('rot', np.eye(2, dtype=np.float32))).astype(np.float32)
        if 'anchor_mask' not in g:
            am = np.zeros(g['n'], bool); am[g['anchors']] = True; g['anchor_mask'] = am

    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    multi_gpu = (num_gpus >= 2)
    if hasattr(torch.autograd.graph, 'set_warn_on_accumulate_grad_stream_mismatch'):
        torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)
    if multi_gpu:
        dev0 = torch.device('cuda:0')
        dev1 = torch.device('cuda:1')
        gpu_info = f'Dual-GPU (cuda:0, cuda:1) [{torch.cuda.get_device_name(0)} + {torch.cuda.get_device_name(1)}]'
    elif num_gpus == 1:
        dev0 = torch.device('cuda:0')
        dev1 = None
        gpu_info = f'Single GPU (cuda:0) [{torch.cuda.get_device_name(0)}]'
    else:
        dev0 = torch.device('cpu')
        dev1 = None
        gpu_info = 'CPU'

    print(f'\n' + '='*80)
    print(f'  STARTING TRAINING: [{model_tag.upper()}]')
    print(f'  Hardware: {gpu_info}')
    print(f'  Train budget: {len(train_graphs)} graphs | Val budget: {len(val_graphs)} graphs')
    print(f'  Max epochs: {max_epochs} | Early stopping patience: {patience} epochs')
    print('='*80)

    in_dim = train_graphs[0]['x'].shape[1]
    edge_dim = train_graphs[0]['edge_attr'].shape[1]

    model = AnchorGNNLocalizer(in_dim=in_dim, edge_dim=edge_dim,
                               H=C.HIDDEN, enc_layers=C.ENC_LAYERS, refine_steps=C.REFINE_STEPS,
                               variant=variant_name, huber_delta=C.HUBER_DELTA).to(dev0)

    model1 = None
    if multi_gpu:
        import copy
        model1 = copy.deepcopy(model).to(dev1)
        s0 = torch.cuda.Stream(device=dev0)
        s1 = torch.cuda.Stream(device=dev1)

    opt = torch.optim.AdamW(model.parameters(), lr=C.LR, weight_decay=C.WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(train_graphs) / batch_size))
    total_steps = max_epochs * steps_per_epoch
    warmup_steps = min(3 * steps_per_epoch, total_steps // 5)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.02, 0.5 * (1.0 + np.cos(np.pi * progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best_ckpt = f'checkpoints/best_model_{model_tag}.pt'
    final_ckpt = f'checkpoints/final_model_{model_tag}.pt'

    best_val_cert = float('inf')
    best_epoch = 0
    patience_counter = 0
    history = []
    t_start = time.time()
    is_enc_only = (variant_name == 'encoder_only')
    half_bs = max(1, batch_size // 2)

    for ep in range(1, max_epochs + 1):
        model.train()
        random.shuffle(train_graphs)
        tot_loss = 0.0
        t0 = time.time()

        for i in range(0, len(train_graphs), batch_size):
            if multi_gpu:
                chunk0 = train_graphs[i:i + half_bs]
                chunk1 = train_graphs[i + half_bs:i + batch_size]
                B0 = collate(chunk0, dev0)
                if chunk1:
                    B1 = collate(chunk1, dev1)
                    model1.load_state_dict(model.state_dict())
                    with torch.cuda.stream(s0):
                        x0, xA0 = model(B0)
                        loss0 = supervised_loss(x0, B0, C.HUBER_DELTA) if is_enc_only else (
                            supervised_loss(x0, B0, C.HUBER_DELTA) + C.AUX_WEIGHT * supervised_loss(xA0, B0, C.HUBER_DELTA)
                        )
                        opt.zero_grad()
                        loss0.backward()
                    with torch.cuda.stream(s1):
                        x1, xA1 = model1(B1)
                        loss1 = supervised_loss(x1, B1, C.HUBER_DELTA) if is_enc_only else (
                            supervised_loss(x1, B1, C.HUBER_DELTA) + C.AUX_WEIGHT * supervised_loss(xA1, B1, C.HUBER_DELTA)
                        )
                        model1.zero_grad()
                        loss1.backward()
                    torch.cuda.synchronize(dev0)
                    torch.cuda.synchronize(dev1)
                    for p0, p1 in zip(model.parameters(), model1.parameters()):
                        if p1.grad is not None:
                            p0.grad.add_(p1.grad.to(dev0))
                            p0.grad.mul_(0.5)
                    step_loss = (loss0.item() + loss1.item()) / 2.0
                    del loss0, loss1, x0, xA0, x1, xA1, B0, B1
                else:
                    x0, xA0 = model(B0)
                    loss0 = supervised_loss(x0, B0, C.HUBER_DELTA) if is_enc_only else (
                        supervised_loss(x0, B0, C.HUBER_DELTA) + C.AUX_WEIGHT * supervised_loss(xA0, B0, C.HUBER_DELTA)
                    )
                    opt.zero_grad()
                    loss0.backward()
                    step_loss = loss0.item()
                    del loss0, x0, xA0, B0
                torch.nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
                opt.step()
                sched.step()
                tot_loss += step_loss
            else:
                chunk = train_graphs[i:i + batch_size]
                B = collate(chunk, dev0)
                x, xA = model(B)
                loss = supervised_loss(x, B, C.HUBER_DELTA) if is_enc_only else (
                    supervised_loss(x, B, C.HUBER_DELTA) + C.AUX_WEIGHT * supervised_loss(xA, B, C.HUBER_DELTA)
                )
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
                opt.step()
                sched.step()
                tot_loss += loss.item()

        avg_train_loss = tot_loss / steps_per_epoch
        v_cert, v_mean, v_med = evaluate_fast(model, val_graphs, dev0, refine_steps=C.TEST_REFINE_STEPS,
                                              model1=model1, dev1=dev1)
        curr_lr = opt.param_groups[0]['lr']

        is_best = False
        if v_cert < (best_val_cert - min_delta):
            best_val_cert = v_cert
            best_epoch = ep
            patience_counter = 0
            is_best = True
            ckpt_payload = dict(
                state=model.state_dict(), epoch=ep, best_val_cert=v_cert,
                variant=variant_name, loss='supervised', train_on=train_on,
                huber_delta=C.HUBER_DELTA, in_dim=in_dim, edge_dim=edge_dim,
                H=C.HIDDEN, enc_layers=C.ENC_LAYERS, refine_steps=C.REFINE_STEPS
            )
            # 1. Standard name for step3_evaluate.py in colab_all_in_one:
            torch.save(ckpt_payload, f'checkpoints/{model_tag}.pt')
            with open(f'checkpoints/{model_tag}.done', 'w') as f:
                f.write(f'{v_cert:.6f}\n')
            # 2. Multi-seed directory (seed0) so colab_all_in_one multi-seed loop immediately skips Step 2:
            os.makedirs('checkpoints/seed0', exist_ok=True)
            torch.save(ckpt_payload, f'checkpoints/seed0/{model_tag}.pt')
            with open(f'checkpoints/seed0/{model_tag}.done', 'w') as f:
                f.write(f'{v_cert:.6f}\n')
            # 3. Explicit alias:
            torch.save(ckpt_payload, best_ckpt)
        else:
            patience_counter += 1

        flag = '  *BEST SAVED*' if is_best else ''
        print(f'  Epoch {ep:2d}/{max_epochs} | Train Loss: {avg_train_loss:.5f} | '
              f'Val Cert RMSE: {v_cert:.4f}m | Mean: {v_mean:.4f}m | '
              f'LR: {curr_lr:.2e} | {time.time()-t0:.1f}s{flag}', flush=True)

        history.append(dict(
            epoch=ep, train_loss=avg_train_loss, val_rmse_cert=v_cert,
            val_rmse_mean=v_mean, val_rmse_median=v_med, lr=curr_lr,
            epoch_time_sec=time.time() - t0
        ))

        if patience_counter >= patience:
            print(f'\n[+] Early stopping triggered for [{model_tag}]! '
                  f'Certified validation RMSE reached minimum at Epoch {best_epoch} ({best_val_cert:.4f}m) '
                  f'and plateaued for {patience} epochs.')
            break

    total_time = time.time() - t_start
    torch.save(dict(state=model.state_dict(), variant=variant_name, train_on=train_on), final_ckpt)
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(f'results/history_{model_tag}.csv', index=False)

    summary = dict(
        variant=model_tag,
        train_graphs=len(train_graphs),
        val_graphs=len(val_graphs),
        converged_epoch=best_epoch,
        total_epochs=len(df_hist),
        cumulative_graphs_to_converge=best_epoch * len(train_graphs),
        best_val_cert_rmse=round(best_val_cert, 4),
        final_train_loss=round(df_hist['train_loss'].iloc[-1], 6),
        train_time_sec=round(total_time, 1)
    )
    print(f'[+] Model [{model_tag}] finished in {total_time:.1f}s. Converged at Epoch {best_epoch} (Val Cert: {best_val_cert:.4f}m)')
    return dict(summary=summary, history_df=df_hist, best_ckpt=best_ckpt)


## Step 4: Train ALL 5 Model Ablation Variants Sequentially
Iterates over `VARIANTS = ['learned', 'uniform', 'huber', 'huber_alpha', 'encoder_only']` on the 6,000 training graphs.
For each variant, weights and convergence logs are saved.


In [ ]:
# Pre-cache canonical coordinates for train and val graphs to eliminate CPU bottlenecks
for g in train_g + val_g + test_g:
    if 'y_norm' not in g:
        g['y_norm'] = (((g['true_xy'] - g['center']) / g['scale']) @ g.get('rot', np.eye(2, dtype=np.float32))).astype(np.float32)
    if 'anchor_mask' not in g:
        am = np.zeros(g['n'], bool); am[g['anchors']] = True; g['anchor_mask'] = am

all_variant_histories = {}
convergence_summary_records = []

device_str = 'cuda:0' if torch.cuda.is_available() else 'cpu'
t_suite_start = time.time()

for var in VARIANTS:
    res = train_variant(
        variant_name=var,
        train_graphs=train_g,
        val_graphs=val_g,
        device=device_str,
        max_epochs=MAX_EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=MIN_DELTA,
        batch_size=BATCH_GRAPHS
    )
    all_variant_histories[var] = res['history_df']
    convergence_summary_records.append(res['summary'])

df_convergence_summary = pd.DataFrame(convergence_summary_records)
df_convergence_summary.to_csv('results/ablation_convergence_summary.csv', index=False)
print(f'\n[+] ALL {len(VARIANTS)} VARIANTS TRAINED in {time.time()-t_suite_start:.1f}s!')
display(df_convergence_summary)


## Step 5: Test Set Evaluation on 750 Held-Out Graphs
Evaluates each best checkpoint on the 750 test graphs across missingness levels ($10\% - 90\%$):
- **Raw Model RMSE** (Stage-A for `encoder_only`, unrolled SMACOF for others)
- **Refined RMSE with Huber Polish** (`+ Huber polish`)
- **Certified Node RMSE** (strictly identifiable nodes)
- **Localization Success Rate** (% of graphs with RMSE < 1.0 m)
- **Inference Speed** (milliseconds per graph)


In [ ]:
print(f'\nEvaluating all {len(VARIANTS)} variants on 750 test graphs...')
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
multi_gpu = (num_gpus >= 2)
dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dev1 = torch.device('cuda:1') if multi_gpu else None
if multi_gpu:
    print(f'[+] Dual-GPU active for evaluation: utilizing GPU 0 and GPU 1 concurrently!')
test_eval_records = []
test_summary_records = []

for var in VARIANTS:
    best_ckpt_path = f'checkpoints/best_model_{var}.pt'
    ckpt_data = torch.load(best_ckpt_path, map_location=dev)
    in_dim = test_g[0]['x'].shape[1]
    edge_dim = test_g[0]['edge_attr'].shape[1]
    model = AnchorGNNLocalizer(in_dim=in_dim, edge_dim=edge_dim,
                               H=C.HIDDEN, enc_layers=C.ENC_LAYERS, refine_steps=C.REFINE_STEPS,
                               variant=var, huber_delta=C.HUBER_DELTA).to(dev)
    model.load_state_dict(ckpt_data['state'])
    model.eval()
    model1 = None
    if multi_gpu:
        import copy
        model1 = copy.deepcopy(model).to(dev1)
        model1.eval()

    t_infer_start = time.perf_counter()
    T_steps = None if var == 'encoder_only' else C.TEST_REFINE_STEPS

    def eval_test_chunk(m, chk, d):
        records, raws, certs, pols = [], [], [], []
        with torch.no_grad():
            for i in range(0, len(chk), 64):
                chunk = chk[i:i + 64]
                B = collate(chunk, d)
                x_pred, _ = m(B, T=T_steps)
                coords_pred = to_original_units(B, x_pred)
                for g, X_gnn in zip(chunk, coords_pred):
                    r_raw = coord_rmse(g, X_gnn)
                    r_cert = coord_rmse(g, X_gnn, g.get('certified'))
                    raws.append(r_raw)
                    certs.append(r_cert)
                    r_pol = np.nan
                    if EVALUATE_HUBER_POLISH:
                        X_pol = polish_coordinates(g, X_gnn)
                        r_pol = coord_rmse(g, X_pol)
                        pols.append(r_pol)
                    records.append(dict(
                        variant=var,
                        scenario=g.get('scenario', 'mixed'),
                        missing=int(round(100 * g['missing_nominal'])),
                        rmse_raw=r_raw,
                        rmse_cert=r_cert,
                        rmse_polished=r_pol,
                        success=bool(r_raw < 1.0)
                    ))
        return records, raws, certs, pols

    if multi_gpu and len(test_g) >= 2:
        import concurrent.futures
        mid = len(test_g) // 2
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            f0 = pool.submit(eval_test_chunk, model, test_g[:mid], dev)
            f1 = pool.submit(eval_test_chunk, model1, test_g[mid:], dev1)
            rec0, raw0, c0, p0 = f0.result()
            rec1, raw1, c1, p1 = f1.result()
            test_eval_records.extend(rec0 + rec1)
            raw_rmses = raw0 + raw1
            cert_rmses = c0 + c1
            polished_rmses = p0 + p1
    else:
        rec, raw_rmses, cert_rmses, polished_rmses = eval_test_chunk(model, test_g, dev)
        test_eval_records.extend(rec)

    infer_duration_ms = (time.perf_counter() - t_infer_start) / len(test_g) * 1000.0
    mean_raw = float(np.nanmean(raw_rmses))
    mean_cert = float(np.nanmean(cert_rmses))
    mean_pol = float(np.nanmean(polished_rmses)) if polished_rmses else np.nan
    succ_rate = float(np.mean([r < 1.0 for r in raw_rmses])) * 100.0

    test_summary_records.append(dict(
        variant=var,
        test_rmse_raw=round(mean_raw, 4),
        test_rmse_certified=round(mean_cert, 4),
        test_rmse_polished=round(mean_pol, 4),
        success_rate_pct=round(succ_rate, 2),
        inference_speed_ms=round(infer_duration_ms, 2)
    ))
    print(f'  Variant [{var:12s}] -> Raw Test RMSE: {mean_raw:.4f}m | Polished: {mean_pol:.4f}m | Cert: {mean_cert:.4f}m | Speed: {infer_duration_ms:.2f}ms/graph')

df_test_eval = pd.DataFrame(test_eval_records)
df_test_eval.to_csv('results/ablation_test_predictions.csv', index=False)

df_test_summary = pd.DataFrame(test_summary_records)
df_test_summary.to_csv('results/ablation_test_summary.csv', index=False)
print('\n[+] Test Set Evaluation Summary:')
display(df_test_summary)


## Step 6: Multi-Variant Comparison Plots & Underlying CSV Exports
Generates 4 publication figures comparing all 5 variants:
1. **`plots/plot1_loss_comparison.png` & `.eps`:** Training loss convergence trajectories (semi-log).
2. **`plots/plot2_val_rmse_convergence.png` & `.eps`:** Certified validation RMSE convergence curves with marked convergence points.
3. **`plots/plot3_test_rmse_vs_missing.png` & `.eps`:** Test coordinate RMSE across missing levels $10\% - 90\%$ (raw vs. polished).
4. **`plots/plot4_ablation_summary_bars.png` & `.eps`:** Grouped bar chart comparing Best Val RMSE, Test RMSE, Convergence Epoch, and Inference Time.

All figures are saved as 300 DPI `.png` and vector `.eps` files, along with the underlying CSV tables.


In [ ]:
import matplotlib.pyplot as plt

COLOR_MAP = {
    'learned':     '#1f77b4',  # Blue (Proposed)
    'uniform':     '#ff7f0e',  # Orange
    'huber':       '#2ca02c',  # Green
    'huber_alpha': '#d62728',  # Red
    'encoder_only':'#9467bd',  # Purple
}

LABEL_MAP = {
    'learned':     'Proposed (Learned gate + learned alpha)',
    'uniform':     'Uniform weights (dwMDS)',
    'huber':       'Huber-IRLS weights (fixed alpha)',
    'huber_alpha': 'Huber-IRLS + learned alpha',
    'encoder_only':'Stage-A GNN only (T=0)',
}

# ---------------- 1. Training Loss Trajectories ----------------
plt.figure(figsize=(9, 5))
loss_export = {'epoch': list(range(1, MAX_EPOCHS + 1))}
for var in VARIANTS:
    df_h = all_variant_histories[var]
    plt.plot(df_h['epoch'], df_h['train_loss'], color=COLOR_MAP[var], lw=2.0, label=LABEL_MAP[var])
    loss_export[f'{var}_loss'] = list(df_h['train_loss']) + [np.nan]*(MAX_EPOCHS - len(df_h))

plt.yscale('log')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Supervised Huber Loss (log scale)', fontsize=12)
plt.title('Training Loss Convergence Across Model Variants (6,000 Graphs)', fontsize=13, fontweight='bold')
plt.grid(True, which='both', ls='--', alpha=0.3)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.savefig('plots/plot1_loss_comparison.png', dpi=300)
plt.savefig('plots/plot1_loss_comparison.eps', format='eps')
plt.show()
pd.DataFrame(loss_export).dropna(how='all').to_csv('results/plot_data_loss_curves.csv', index=False)

# ---------------- 2. Validation RMSE Convergence Curves ----------------
plt.figure(figsize=(9, 5))
val_export = {'epoch': list(range(1, MAX_EPOCHS + 1))}
for var in VARIANTS:
    df_h = all_variant_histories[var]
    best_row = df_h.loc[df_h['val_rmse_cert'].idxmin()]
    line, = plt.plot(df_h['epoch'], df_h['val_rmse_cert'], color=COLOR_MAP[var], lw=2.0, label=LABEL_MAP[var])
    plt.scatter([best_row['epoch']], [best_row['val_rmse_cert']], color=COLOR_MAP[var], s=80, zorder=5)
    val_export[f'{var}_val_rmse_cert'] = list(df_h['val_rmse_cert']) + [np.nan]*(MAX_EPOCHS - len(df_h))

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Certified Validation Coordinate RMSE (m)', fontsize=12)
plt.title('Validation Error Convergence Curves (Dots = Minimum / Best Checkpoint)', fontsize=13, fontweight='bold')
plt.grid(True, ls='--', alpha=0.3)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.savefig('plots/plot2_val_rmse_convergence.png', dpi=300)
plt.savefig('plots/plot2_val_rmse_convergence.eps', format='eps')
plt.show()
pd.DataFrame(val_export).dropna(how='all').to_csv('results/plot_data_val_convergence.csv', index=False)

# ---------------- 3. Test RMSE vs Missing Distance % ----------------
piv_test = df_test_eval.pivot_table(index='missing', columns='variant', values=['rmse_raw', 'rmse_polished'], aggfunc='mean')
plt.figure(figsize=(10, 5))
for var in VARIANTS:
    plt.plot(piv_test.index, piv_test[('rmse_raw', var)], color=COLOR_MAP[var], marker='o', lw=2.0, label=f'{var} (Raw)')
    if EVALUATE_HUBER_POLISH:
        plt.plot(piv_test.index, piv_test[('rmse_polished', var)], color=COLOR_MAP[var], marker='s', lw=1.5, ls='--', alpha=0.7, label=f'{var} (+ Huber polish)')

plt.xlabel('Missing Distance Measurements (%)', fontsize=12)
plt.ylabel('Test Coordinate RMSE (m)', fontsize=12)
plt.title('Test Generalization Under Severe Anchor Scarcity (10% to 90% Missing)', fontsize=13, fontweight='bold')
plt.grid(True, ls='--', alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('plots/plot3_test_rmse_vs_missing.png', dpi=300)
plt.savefig('plots/plot3_test_rmse_vs_missing.eps', format='eps')
plt.show()
piv_test.to_csv('results/plot_data_test_vs_missing.csv')

# ---------------- 4. Multi-Metric Grouped Bar Chart ----------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
master_df = df_convergence_summary.merge(df_test_summary, on='variant')
master_df.to_csv('results/ablation_master_summary.csv', index=False)

bar_colors = [COLOR_MAP[v] for v in master_df['variant']]
axes[0].bar(master_df['variant'], master_df['converged_epoch'], color=bar_colors, alpha=0.85)
axes[0].set_title('Convergence Epoch ($E_{conv}$)', fontweight='bold')
axes[0].set_ylabel('Epochs')

axes[1].bar(master_df['variant'], master_df['best_val_cert_rmse'], color=bar_colors, alpha=0.85)
axes[1].set_title('Best Val Cert RMSE', fontweight='bold')
axes[1].set_ylabel('RMSE (meters)')

axes[2].bar(master_df['variant'], master_df['test_rmse_raw'], color=bar_colors, alpha=0.85)
axes[2].set_title('Test RMSE (Raw Model)', fontweight='bold')
axes[2].set_ylabel('RMSE (meters)')

axes[3].bar(master_df['variant'], master_df['inference_speed_ms'], color=bar_colors, alpha=0.85)
axes[3].set_title('Inference Speed', fontweight='bold')
axes[3].set_ylabel('Time (ms / graph)')

for ax in axes:
    ax.grid(axis='y', ls='--', alpha=0.3)
    ax.set_xticklabels(master_df['variant'], rotation=25, ha='right', fontsize=9)

plt.suptitle('Ablation Study Summary: Performance, Convergence & Efficiency Trade-offs', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plots/plot4_ablation_summary_bars.png', dpi=300)
plt.savefig('plots/plot4_ablation_summary_bars.eps', format='eps')
plt.show()


## Step 7: Convergence Diagnosis & Production Recommendations
Consolidates exact convergence epochs, cumulative training graph counts, and percentage error improvements to use directly in `config.py`.


In [ ]:
print('='*85)
print('               ABLATION STUDY MASTER RESULTS TABLE')
print('='*85)
display(master_df[['variant', 'converged_epoch', 'cumulative_graphs_to_converge',
                   'best_val_cert_rmse', 'test_rmse_raw', 'test_rmse_polished',
                   'success_rate_pct', 'inference_speed_ms', 'train_time_sec']])

# Extract key statistics
learned_row = master_df[master_df.variant == 'learned'].iloc[0]
uniform_row = master_df[master_df.variant == 'uniform'].iloc[0]
huber_row   = master_df[master_df.variant == 'huber'].iloc[0]
enc_row     = master_df[master_df.variant == 'encoder_only'].iloc[0]

imp_over_uniform = (uniform_row['test_rmse_raw'] - learned_row['test_rmse_raw']) / uniform_row['test_rmse_raw'] * 100.0
imp_over_huber   = (huber_row['test_rmse_raw'] - learned_row['test_rmse_raw']) / huber_row['test_rmse_raw'] * 100.0
imp_over_enc     = (enc_row['test_rmse_raw'] - learned_row['test_rmse_raw']) / enc_row['test_rmse_raw'] * 100.0

print('='*85)
print('            EXACT PRODUCTION RECOMMENDATIONS FOR MAIN CODE')
print('='*85)
print(f"1. OPTIMAL NUMBER OF EPOCHS:")
print(f"   - Proposed model 'learned' converged at Epoch {learned_row['converged_epoch']} (out of {MAX_EPOCHS}).")
print(f"   - Maximum convergence epoch across all variants: {master_df['converged_epoch'].max()}.")
print(f"   - Recommended production setting in 'config.py': EPOCHS = {int(np.ceil(master_df['converged_epoch'].max() * 1.15))} (includes 15% safety margin).")

print(f"\n2. TRAINING DATASET BUDGET:")
print(f"   - The model converged stably after processing {learned_row['cumulative_graphs_to_converge']:,} training graph passes.")
print(f"   - Using 6,000 graphs (with 750 val graphs) provides sufficient statistical diversity.")

print(f"\n3. ABLATION CONCLUSION FOR IEEE SPL:")
print(f"   - Proposed Learned SMACOF reduces test error by {imp_over_uniform:.1f}% compared to uniform dwMDS.")
print(f"   - Proposed Learned SMACOF reduces test error by {imp_over_huber:.1f}% compared to classical Huber IRLS.")
print(f"   - Unrolled refinement reduces test error by {imp_over_enc:.1f}% compared to pure Stage-A GNN.")
print('='*85)


## Step 8: Cross-Topology Transfer Learning Experiment (From `colab_all_in_one.ipynb`)
This experiment is **fundamentally different** from the Step 4 ablation study:
- **Step 4 (Ablation):** Compares 5 model architectures on mixed graphs to prove neural gating works.
- **Step 8 (Transfer):** Keeps the model architecture fixed (`learned`), but trains 7 separate models on isolated graph subsets:
  1. `random`: Trained ONLY on Random Spanning Tree graphs (never sees RGG or BA).
  2. `rgg`: Trained ONLY on Random Geometric Graphs (never sees Random or BA).
  3. `ba`: Trained ONLY on Barabási-Albert scale-free networks (never sees Random or RGG).
  4. `random+rgg`: Trained on pairs (never sees BA).
  5. `random+ba`: Trained on pairs (never sees RGG).
  6. `rgg+ba`: Trained on pairs (never sees Random).
  7. `all`: Trained on all 3 (same total budget split equally across types).

All 7 models are evaluated across all test topologies (for missing $\ge 50\%$), producing the **IEEE SPL Cross-Topology Transfer Matrix**.


In [ ]:
# Cross-Topology Transfer Experiment: Train on subsets, test on all
import pandas as pd
import torch
from IPython.display import display
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
multi_gpu = (num_gpus >= 2)
dev0 = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dev1 = torch.device('cuda:1') if multi_gpu else None
TRANSFER_SETS = ['random', 'rgg', 'ba', 'random+rgg', 'random+ba', 'rgg+ba', 'all']
transfer_eval_records = []

if RUN_TRANSFER_TEST:
    print('\n' + '='*80)
    print('  STARTING CROSS-TOPOLOGY TRANSFER EXPERIMENT (7 MODELS)')
    if multi_gpu:
        print(f'  Dual-GPU Active: Utilizing both GPU 0 and GPU 1 for transfer training and testing!')
    print('='*80)
    for t_set in TRANSFER_SETS:
        tag_name = f'learned__train-{t_set}'
        done_path = f'checkpoints/{tag_name}.done'
        done_seed0 = f'checkpoints/seed0/{tag_name}.done'
        if os.path.exists(done_path) or os.path.exists(done_seed0):
            print(f'  [+] Skip transfer {t_set}: already trained (.done exists)')
        else:
            train_variant(
                variant_name='learned',
                train_on=t_set,
                max_epochs=MAX_EPOCHS,
                patience=EARLY_STOPPING_PATIENCE,
                min_delta=MIN_DELTA,
                batch_size=BATCH_GRAPHS
            )

    print('\n[+] Evaluating all 7 transfer models across test scenarios (missing >= 50%)...')
    with open('data/test.pkl', 'rb') as f: test_graphs = pickle.load(f)
    d_test_50 = [g for g in test_graphs if g['missing_nominal'] >= 0.50]

    for t_set in TRANSFER_SETS:
        tag_name = f'learned__train-{t_set}'
        ckpt_f = f'checkpoints/{tag_name}.pt' if os.path.exists(f'checkpoints/{tag_name}.pt') else f'checkpoints/seed0/{tag_name}.pt'
        ck = torch.load(ckpt_f, map_location=dev0)
        m = AnchorGNNLocalizer(ck['in_dim'], ck['edge_dim'], ck['H'],
                               ck['enc_layers'], ck['refine_steps'], ck['variant'],
                               huber_delta=ck.get('huber_delta', C.HUBER_DELTA)).to(dev0)
        m.load_state_dict(ck['state'])
        m.eval()
        m1 = None
        if multi_gpu:
            import copy
            m1 = copy.deepcopy(m).to(dev1)
            m1.eval()

        def eval_transfer_chunk(model_obj, chk_graphs, dev_t):
            chunk_records = []
            with torch.no_grad():
                for i in range(0, len(chk_graphs), 64):
                    chk = chk_graphs[i:i + 64]
                    B = collate(chk, dev_t)
                    x_pred, _ = model_obj(B, T=C.TEST_REFINE_STEPS)
                    preds = to_original_units(B, x_pred)
                    for g, X_gnn in zip(chk, preds):
                        r = coord_rmse(g, X_gnn)
                        chunk_records.append(dict(
                            method=f'Ours, trained on {t_set}',
                            train_on=t_set,
                            scenario=g['scenario'],
                            missing=int(round(100 * g['missing_nominal'])),
                            rmse=r,
                        ))
                        if EVALUATE_HUBER_POLISH:
                            X_pol = polish_coordinates(g, X_gnn)
                            chunk_records.append(dict(
                                method=f'Ours, trained on {t_set} + Huber polish',
                                train_on=t_set,
                                scenario=g['scenario'],
                                missing=int(round(100 * g['missing_nominal'])),
                                rmse=coord_rmse(g, X_pol),
                            ))
            return chunk_records

        if multi_gpu and len(d_test_50) >= 2:
            import concurrent.futures
            mid = len(d_test_50) // 2
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
                f0 = pool.submit(eval_transfer_chunk, m, d_test_50[:mid], dev0)
                f1 = pool.submit(eval_transfer_chunk, m1, d_test_50[mid:], dev1)
                transfer_eval_records.extend(f0.result() + f1.result())
        else:
            transfer_eval_records.extend(eval_transfer_chunk(m, d_test_50, dev0))

    df_trans = pd.DataFrame(transfer_eval_records)
    df_trans.to_csv('results/transfer_results.csv', index=False)

    # Pivot Table: Rows = Method (Training Set), Columns = Scenario
    piv = df_trans.pivot_table(index='method', columns='scenario', values='rmse', aggfunc='mean')
    piv.to_csv('results/transfer_matrix.csv')
    print('\n=== CROSS-TOPOLOGY TRANSFER MATRIX (Mean RMSE, Missing >= 50%) ===')
    display(piv.round(3))

    # Publication Heatmap
    plt.figure(figsize=(10, 6))
    im = plt.imshow(piv.values, cmap='Blues_r', aspect='auto')
    plt.colorbar(im, label='Test Coordinate RMSE (m, lower is better)')
    plt.xticks(range(piv.shape[1]), piv.columns, rotation=30, ha='right')
    plt.yticks(range(piv.shape[0]), piv.index)
    for r in range(piv.shape[0]):
        for c in range(piv.shape[1]):
            val = piv.values[r, c]
            if not np.isnan(val):
                col = 'white' if val > piv.values[~np.isnan(piv.values)].mean() else 'black'
                plt.text(c, r, f'{val:.3f}', ha='center', va='center', color=col, fontsize=8)
    plt.title('Cross-Topology Transfer Learning Matrix (AnchorGNN + Learned SMACOF)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/fig_transfer_matrix.png', dpi=300)
    plt.savefig('plots/fig_transfer_matrix.eps', format='eps')
    plt.show()


## Step 9: Kaggle Output Verification & Master Results Archive
This cell packages all outputs into a single zip archive and copies key CSVs and plots to the root of `/kaggle/working`.

> [!IMPORTANT]
> **How Kaggle Output Works:**
> Everything saved in `/kaggle/working` is automatically indexed by Kaggle.
> When the notebook finishes or when you click **Save Version**, look at the right sidebar under **Data > Output**.
> You will see `gnn_loc_all_variants_6000graphs.zip`, `ablation_master_summary.csv`, and all generated plots ready for 1-click download!


In [ ]:
# Copy key master CSVs and figures to the root of WORKDIR for direct visibility in Kaggle Output
for f in ['results/ablation_master_summary.csv', 'results/ablation_test_summary.csv',
          'results/transfer_matrix.csv', 'results/transfer_results.csv',
          'plots/plot1_loss_comparison.png', 'plots/plot2_val_rmse_convergence.png',
          'plots/plot3_test_rmse_vs_missing.png', 'plots/plot4_ablation_summary_bars.png',
          'plots/fig_transfer_matrix.png']:
    if os.path.exists(f):
        dest = os.path.join(WORKDIR, os.path.basename(f))
        if os.path.abspath(f) != os.path.abspath(dest):
            shutil.copy2(f, dest)

# Create Master Zip Archive using zipfile
import zipfile
zip_name = 'gnn_loc_all_variants_6000graphs.zip'
zip_path = os.path.join(WORKDIR, zip_name)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root_dir in ['checkpoints', 'results', 'plots', 'data']:
        full_dir = os.path.join(WORKDIR, root_dir)
        if os.path.exists(full_dir):
            for root, _, files in os.walk(full_dir):
                for f in files:
                    abs_f = os.path.join(root, f)
                    rel_f = os.path.relpath(abs_f, WORKDIR)
                    zf.write(abs_f, rel_f)
    for single_f in ['config.py', 'common.py', 'model.py', 'ablation_master_summary.csv', 'ablation_test_summary.csv']:
        full_f = os.path.join(WORKDIR, single_f)
        if os.path.exists(full_f):
            zf.write(full_f, single_f)

print('='*80)
print('                KAGGLE OUTPUT FILES CONFIRMATION')
print('='*80)
print(f'Root Output Directory: {WORKDIR}')
total_size_mb = 0.0
for item in sorted(os.listdir(WORKDIR)):
    item_path = os.path.join(WORKDIR, item)
    if os.path.isfile(item_path):
        sz = os.path.getsize(item_path) / (1024 * 1024)
        total_size_mb += sz
        print(f'  [FILE] {item:40s} : {sz:6.2f} MB')
    elif os.path.isdir(item_path) and not item.startswith('.'):
        count = len(os.listdir(item_path))
        print(f'  [DIR ] {item:40s} : ({count} items inside)')

print(f'\n[+] Master Results Zip Archive created: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)')
print('[+] In Kaggle: Look at the right sidebar under Data > Output to view and download all files!')

if IS_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
        print('[+] Automatic Colab download triggered!')
    except Exception:
        pass
print('='*80)


## Step 10: Seamless Transfer to `colab_all_in_one.ipynb` (Run Step 3 Directly)
Once training is finished here, all model checkpoints (`learned.pt`, `uniform.pt`, `huber.pt`, `huber_alpha.pt`, `encoder_only.pt`) and their completion tokens (`.done`) are saved in both `checkpoints/` and `checkpoints/seed0/`.

> [!TIP]
> **How to run Step 3 in `colab_all_in_one.ipynb` without retraining:**
> 1. If you share or copy the `checkpoints/` and `data/` folders to your `colab_all_in_one.ipynb` working directory,
> 2. Running Cell 12 (Step 1) will detect that `data/train.pkl` and `data/test.pkl` already exist and will **skip Step 1**.
> 3. Running Cell 14 (Step 2) will detect that all `.done` files exist (`checkpoints/seed0/*.done`) and will **skip Step 2 completely**.
> 4. Running Cell 16 (Step 3) will execute `python step3_evaluate.py`, which loads the frozen weights, evaluates all 5 GNN models and all baseline algorithms on identical test graphs, computes Wilcoxon significance tests, and generates all paper figures!

The cell below verifies that all checkpoints and `.done` files are in place and ready for direct Step 3 execution.


In [ ]:
# Verify Checkpoints and Dataset readiness for Step 3
print('='*75)
print('       VERIFYING READINESS FOR DIRECT STEP 3 EVALUATION')
print('='*75)

# 1. Verify Dataset files
data_files = ['train.pkl', 'val.pkl', 'test.pkl']
missing_data = [f for f in data_files if not os.path.exists(f'data/{f}')]
if not missing_data:
    print('[+] Dataset files in data/: READY (train, val, test exist)')
else:
    print(f'[-] Warning: Missing data files: {missing_data}')

# 2. Verify Checkpoints and .done markers
missing_ckpts = []
for var in VARIANTS:
    pt_ok = os.path.exists(f'checkpoints/{var}.pt') or os.path.exists(f'checkpoints/seed0/{var}.pt')
    done_ok = os.path.exists(f'checkpoints/{var}.done') or os.path.exists(f'checkpoints/seed0/{var}.done')
    status = 'READY' if (pt_ok and done_ok) else 'MISSING'
    print(f'  Model [{var:12s}] : {status} (pt={pt_ok}, done={done_ok})')
    if not (pt_ok and done_ok):
        missing_ckpts.append(var)

if not missing_ckpts and not missing_data:
    print('\n[+] ALL MODELS & DATASETS ARE READY FOR DIRECT STEP 3 EVALUATION!')
    print('[+] If you switch to colab_all_in_one.ipynb, Step 2 will be completely skipped and Step 3 will run immediately.')
else:
    print('\n[-] Some models have not completed training yet. Run Step 4 first.')
print('='*75)
